## 1. Setup and Configuration
First, we import the necessary libraries and configure our connection to the LLM backend (SambaNova or Ollama) and our embedding model. Adjust the `BACKEND` variables here to switch between local and cloud models.

In [1]:

from openai import OpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_sambanova import SambaNovaEmbeddings
import os
from langchain_ollama import OllamaEmbeddings

# load .env file from last demo
from dotenv import load_dotenv
load_dotenv()  # Loads .env into os.environ

# --- LLM Backend Configuration ---
# Un‑comment one block at a time.

# (1) SambaNova
# BACKEND = "sambanova"
# CHAT_MODEL = "Llama-4-Maverick-17B-128E-Instruct"
# BASE_URL = "https://ai.tejas.tacc.utexas.edu"
# API_KEY = os.getenv("API_KEY")
# EMBEDDING_MODEL = "E5-Mistral-7B-Instruct"
# os.environ["SAMBANOVA_API_BASE"] = BASE_URL
# os.environ["SAMBANOVA_API_KEY"] = API_KEY


# (2) Ollama (OpenAI‑compatible API)
BACKEND = "ollama"
CHAT_MODEL = "llama3.2"  # or whatever model you run locally
BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"  # any dummy string; Ollama ignores it
EMBEDDING_MODEL = "nomic-embed-text"

# RAG parameters
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
MAX_NEW_TOKENS = 512
NUM_K = 3

## 2. Defining the LLM Call
Here we set up our OpenAI-compatible client. The `llm_invoke` function is a helper that takes a formatted prompt, sends it to the selected LLM, and returns the generated text.

In [2]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def llm_invoke(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_NEW_TOKENS,
        temperature=0.2,
    )
    return resp.choices[0].message.content

## 3. Ingesting Documents & Building the Vector Database
This step only needs to be run once. We load all PDFs from the `./docs/` folder, split them into smaller chunks of text, and convert those chunks into mathematical vectors (embeddings). Finally, we store them in a Chroma database.

In [3]:
print("Loading documents from ./docs/ ...")
loader = DirectoryLoader(
    "./docs/",
    glob="**/*.pdf",
    show_progress=False,
    loader_cls=PyPDFLoader,
    load_hidden=False,
)
docs = loader.load()

print(f"Loaded {len(docs)} document pages. Chunking text...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
texts = text_splitter.split_documents(docs)

print("Generating embeddings and initializing Chroma DB...")
# embeddings = SambaNovaEmbeddings(
#     model=EMBEDDING_MODEL,
#     batch_size=16,
# )

# UN-comment 3 lines below if you want to use OLLAMA
embeddings = OllamaEmbeddings(
   model=EMBEDDING_MODEL,
)

db = Chroma.from_documents(
    texts,
    embeddings,
    persist_directory="db_ce",
)
print("Database is ready to be queried!")

Loading documents from ./docs/ ...
Loaded 76 document pages. Chunking text...
Generating embeddings and initializing Chroma DB...
Database is ready to be queried!


## 4. The RAG Query Function
Now we define a reusable function, `ask_rag_bot`. It takes a user question and the database we just built, searches for the most relevant document chunks, and passes both the question and the context to the LLM to generate an informed answer.

In [4]:
def ask_rag_bot(question: str, vector_db):
    """
    Searches the vector database for context and uses the LLM to answer the question.
    """
    # 1. Retrieve the top K most similar document chunks
    results = vector_db.similarity_search(question, k=NUM_K)
    retrieved_context = "\n\n".join(result.page_content for result in results)

    # 2. Format the prompt with the question and the retrieved context
    prompt = ChatPromptTemplate.from_template(
        """You are a helpful AI assistant.
            Use the following retrieved context to help answer the question.
            Keep your answers concise. Two sentences maximum.

            Question: {question}

            Retrieved Context:
            {context}
            """
    )
   
    formatted_prompt = prompt.format(question=question, context=retrieved_context)

    # 3. Ask the LLM
    response = llm_invoke(formatted_prompt)

    # 4. Print the output 
    print("\n=====================================================")
    print(f"The Question:\n{question}\n")
    print("The Answer:")
    print(f"{response}\n")
    
    print("Retrieved Reference Documents:")
    for idx, result in enumerate(results):
        # Extract source file from metadata (if available)
        source = result.metadata.get('source', 'Unknown Source')
        page = result.metadata.get('page', 'Unknown Page')
        
        print(f"--- Document {idx + 1} (Source: {source}, Page: {page}) ---")
        # Printing just the first 300 characters of the content
        print(f"{result.page_content[:300]}...\n")
    print("=====================================================\n")

## 5. Ask Questions!
With everything set up, you can now pass any question to your function.

In [7]:
# Define your questions
messages = [
    "Describe Aurora's neural network architecture.",
    "What did the authors use as the training objective?",
]

# Loop through and ask the bot
for q in messages:
    ask_rag_bot(question=q, vector_db=db)

# You can also ask ad-hoc questions directly!
# ask_rag_bot("your-question-here", db)


The Question:
Describe Aurora's neural network architecture.

The Answer:
Aurora's backbone is a 3D Swin Transformer U-Net19,50 architecture, which serves as a neural simulator with efficient multiscale processing capabilities. This architecture features local self-attention operations within windows and a symmetric upsampling-downsampling structure.

Retrieved Reference Documents:
--- Document 1 (Source: docs/manuscript_foundation_model_for_earth_system.pdf, Page: 6) ---
using adaptive Fourier neural operators. In Proc. Platform for Advanced Scientific 
Computing Conference, article no. 13 (Association for Computing Machinery, 2023).
39. DeMaria, M. et al. Evaluation of tropical cyclone track and intensity forecasts from 
Artificial Intelligence Weather Prediction (...

--- Document 2 (Source: docs/manuscript_foundation_model_for_earth_system.pdf, Page: 8) ---
specifics on input processing, pressure-level aggregation and further 
encodings, see Supplementary Information Sections B.1 